# 03 — Two-Stage Default · REG · catboost (raw mode — 전처리 없음)

`zit_only_raw` 패턴을 Stage 2 회귀기에 적용. `preprocess.run()` 생략하고 CatBoost의 NaN 네이티브 처리(`nan_mode='Min'` 기본)에 맡겨 원본 분포 그대로 학습. y>0 die만으로 학습.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/default/reg/catboost_raw/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **전처리**: Stage 0만 적용. cleaning / imputation / outlier winsorize / 상관 제거 전부 SKIP
- **TARGET_TRANSFORM**: `'none'` 고정, **Y_POSITIVE_ONLY=True**
- **HPO**: `N_TRIALS=10000` + `TIMEOUT_SEC=24h` 안전망
- **anchor**: 전처리 버전 anchor를 시작점으로 enqueue

## 1. 환경 설정 + import

In [1]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'   # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'   # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'   # preprocessing.zip (raw는 EXCLUDE_COLS·meta_features만 사용, import 호환용)
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (코드 수정 시 재업로드)
GDRIVE_OUTPUT_ID        = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'   # 4_output.zip = 기존 실험 산출물 (RESUME 시 복원용)
RESUME                  = True   # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../../../../setup.py만 실행
try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    # RESUME이면 이전 4_output을 통째로 복원 (이미 폴더 있으면 skip)
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 추가 — raw는 EXCLUDE_COLS·meta_features만 쓰지만 같은 폴더라 sys.path 추가 필요
PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)
# `from modules import ...`가 3_modeling/modules를 찾게
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

# raw 모드: preprocess.run()은 사용 안 함 — EXCLUDE_COLS(Stage 0 제외 목록)만 가져옴
from modules.preprocess import EXCLUDE_COLS as _WAFER_MAP_EXCLUDE
from modules import hpo, models                    # hpo.run_hpo / refit_best / save_artifacts, models 레지스트리
from meta_features import add_meta_features         # die_xy / position 메타피처 헬퍼

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)   # LGBM 로그 억제
optuna.logging.set_verbosity(optuna.logging.WARNING)    # Optuna 로그 억제

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
optuna v4.7.0


## 2. 실험 설정

In [2]:
# 회귀 모델 고정 (Two-Stage Stage 2 — y>0인 unit만으로 학습 → E[Y|Y>0,x], raw mode)
REG_MODEL_NAME = 'catboost'
assert REG_MODEL_NAME in models.AVAILABLE_MODELS

# 실험 식별 — 출력 폴더/DB 파일명에 들어감 (raw임을 명시)
EXP_ID   = f'ts-reg-{REG_MODEL_NAME}-raw-001'
EXP_MEMO = f'Two-Stage default · REG · {REG_MODEL_NAME} · y>0 conditional · raw mode (preprocess.run 생략, NaN 네이티브 - nan_mode=Min)'
USER     = 'jh'

# Optuna 예산 — 로컬 런스크립트에서 타임아웃으로 컨트롤. trial은 크게, timeout은 안전망
N_TRIALS         = 10000     # 외부 런스크립트 타임아웃이 실제 종료를 결정하므로 충분히 크게 (CatBoost는 시간 우선 배분 권장)
N_FOLDS          = 5
N_STARTUP_TRIALS = 40        # TPE가 학습을 시작하기 전 무작위 trial 수 (CatBoost는 trial당 비용이 커서 lgbm/xgb보다 작게)
N_JOBS           = 10        # 모델 학습 병렬도 (CatBoost는 thread_count로 매핑됨)
TIMEOUT_SEC      = 62 * 60 * 60   # 24h 안전망 — 노트북 단독 실행 시 보호장치

# Stage 2 정책: y>0 die만 학습(Y_POSITIVE_ONLY), target 변환은 'none' 통일
TARGET_TRANSFORM = 'none'       # 트리는 'none' 통일 (log1p와 사실상 동등)
Y_POSITIVE_ONLY  = True         # fit 데이터에서 y==0 die를 빼고 학습 (= E[Y|Y>0,x] 추정)
CLIP_Y_EXTREME   = True         # train y의 max(=1.0, 단 1건)를 두 번째 큰 값으로 clip

# 출력 경로 — 모델명 뒤에 _raw 표기 (기존 비-raw 디렉토리와 충돌 방지)
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'reg', REG_MODEL_NAME, 'raw', EXP_ID.split('-')[-1])
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')   # study가 여기에 자동 저장 (RESUME 시 여기서 이어서)
os.makedirs(OUT_DIR, exist_ok=True)

# raw 모드 — PP_FIXED 없음 (cleaning/imputation/outlier 전부 SKIP, Stage 0만 적용)

# anchor — 전처리 버전 anchor (당시엔 log1p ON 컨텍스트). raw에서는 feature space가 달라 trial 0 시작점으로만 사용 (Optuna가 재탐색)
REG_ANCHOR = {'iterations': 2408, 'learning_rate': 0.212, 'depth': 10, 'l2_leaf_reg': 1.508, 'random_strength': 0.661, 'bagging_temperature': 0.648, 'border_count': 146, 'rsm': 0.325}

print(f'EXP: {EXP_ID} | USER: {USER} | raw_mode=True')
print(f'REG_MODEL_NAME: {REG_MODEL_NAME}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS} | TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | Y_POSITIVE_ONLY={Y_POSITIVE_ONLY}')
print(f'OUT_DIR={OUT_DIR}')

EXP: ts-reg-catboost-raw-001 | USER: jh | raw_mode=True
REG_MODEL_NAME: catboost
N_TRIALS=10000 | N_FOLDS=5 | N_JOBS=10 | TIMEOUT_SEC=223200
TARGET_TRANSFORM=none | Y_POSITIVE_ONLY=True
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\03_two_stage\default\reg\catboost\raw\001


## 3. 데이터 로드 + Y clip + Stage 0 (raw mode, target_transform=none)

`preprocess.run()` 호출 없음. cleaning / spatial imputation / winsorize / 고상관 제거 전부 SKIP. CatBoost는 `nan_mode='Min'`(기본)으로 NaN을 자동 처리하므로 imputation 불필요.

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# 트리는 target 변환 없음 ('none')
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] {TARGET_TRANSFORM}')

# Stage 0만 적용
n_before = len(feat_cols)
feat_cols_raw = [c for c in feat_cols if c not in _WAFER_MAP_EXCLUDE]
print(f'[Stage 0] 웨이퍼맵 사전 제외: {n_before} → {len(feat_cols_raw)} ({n_before - len(feat_cols_raw)}개 제거)')
print('[raw mode] cleaning / imputation / outlier winsorize / 상관 제거 전부 SKIP')

xs_train = xs_dict['train'].copy()
xs_val   = xs_dict['validation'].copy()
xs_test  = xs_dict['test'].copy()

feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_raw,
    position_mode='raw', use_die_xy=True,
)

nan_pct_train = xs_train[feat_cols_clean].isna().to_numpy().mean() * 100
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  NaN in train[feat_cols]: {xs_train[feat_cols_clean].isna().to_numpy().sum():,} ({nan_pct_train:.2f}%) — CatBoost가 직접 처리')

yt = ys_input['train'][TARGET_COL]
print(f'\nUnit y>0 비율 (train): {(yt > 0).mean():.4f} ({(yt > 0).sum():,} unit)')
print(f'E[Y | Y>0]            = {yt[yt > 0].mean():.6f}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[target transform] none
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
[raw mode] cleaning / imputation / outlier winsorize / 상관 제거 전부 SKIP
[add_meta_features] position_mode='raw', use_die_xy=True, use_loc_x_ohe=False → position=['position'], die_xy=['die_x', 'die_y'] (feat_cols: 1034)

[전처리 완료] feat_cols: 1034
  NaN in train[feat_cols]: 1,675,335 (1.55%) — CatBoost가 직접 처리

Unit y>0 비율 (train): 0.2920 (7,646 unit)
E[Y | Y>0]            = 0.008496


## 4. Optuna HPO (REG, y>0 die만 학습)

- `run_hpo(y_positive_only=True, target_transform_fn=None, target_inverse_fn=None)`
- **Sampler/Pruner/Timeout**: §4·§25
- **anchor enqueue**: 1차 best HP (log1p ON 컨텍스트) — 시작점, Optuna가 재탐색
- 손실함수는 anchor에서 빼고 Optuna가 재선택 (log1p OFF 환경)

In [4]:
# study에 박제할 재현성 메타 — 어떤 조건으로 학습됐는지 study DB만 보고도 알 수 있게
study_meta = {
    'exp_id':           EXP_ID,
    'exp_memo':         EXP_MEMO,
    'user':             USER,
    'reg_model_name':   REG_MODEL_NAME,
    'raw_mode':         True,                   # raw임을 명시 (분석/필터링용 마커)
    'target_transform': TARGET_TRANSFORM,
    'y_positive_only':  Y_POSITIVE_ONLY,
    'clip_y_extreme':   CLIP_Y_EXTREME,
    'effective_pp_params': {},                  # raw mode — preprocess.run 미사용 (빈 dict로 박제)
    'n_trials':         N_TRIALS,
    'n_folds':          N_FOLDS,
    'n_jobs':           N_JOBS,
    'timeout_sec':      TIMEOUT_SEC,
    'seed':             SEED,
    'anchor':           REG_ANCHOR,
    'sampler':          'TPE seed=None multivariate group',
    'pruner':           f'MedianPruner n_warmup=10',
}

# TPE: multivariate=HP 결합 분포 학습, group=조건부 축 자동 skip, seed=None → run마다 다양성
sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
# MedianPruner: 처음 10 step은 안 자르고, 그 후 중앙값보다 나쁘면 가지치기
pruner  = MedianPruner(n_warmup_steps=10)

# 회귀 HPO: y_positive_only=True → fit 데이터에서 y==0 die를 빼고 학습 (= E[Y|Y>0,x] 추정), objective는 unit OOF RMSE
# anchor를 trial 0으로 강제(enqueue_trials). val/test RMSE는 매 trial user_attr에 기록
res = hpo.run_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=REG_MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    y_positive_only=Y_POSITIVE_ONLY,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=RESUME,
    user_attrs=study_meta,
    sampler=sampler,
    pruner=pruner,
    enqueue_trials=[REG_ANCHOR],   # anchor를 첫 trial로 (기존 trial 없을 때만)
    timeout=TIMEOUT_SEC,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
study       = res['study']
best_params = res['best_params']

print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'best_params = {best_params}')

[enqueue skip] 기존 trial 15개 — resume 모드


  0%|          | 0/10000 [00:00<?, ?it/s]


[HPO 완료] best OOF RMSE = 0.006953
best_params = {'iterations': 3029, 'learning_rate': 0.06844267887926207, 'depth': 11, 'l2_leaf_reg': 3.2322509052592987, 'random_strength': 0.15398838005913912, 'bagging_temperature': 0.11967654390236546, 'border_count': 224, 'rsm': 0.5650059836662208, 'loss_function': 'Tweedie', 'tweedie_variance_power': 1.8869748405708124}


## 5. Best trial 재학습 (K-fold OOF) + die-level reg_pred 캐쳐

In [5]:
# best HP로 5-fold 재학습 (y>0 die만 학습) → die-level OOF / val / test 예측. (단독 평가용 — combine 전이라 ×P(Y>0) 안 함)
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=REG_MODEL_NAME,
    best_params=best_params,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    y_positive_only=Y_POSITIVE_ONLY,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

# 후처리 이전(mean 집계) unit RMSE를 정답과 정렬해 계산 (Stage 2 단독이라 RMSE 자체는 의미 제한적 — 진짜 평가는 combine에서)
y_train_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_true   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_true  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

oof_unit  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_train_true.index]
val_unit  = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
test_unit = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]

oof_rmse  = float(np.sqrt(np.mean((oof_unit.values  - y_train_true.values) ** 2)))
val_rmse  = float(np.sqrt(np.mean((val_unit.values  - y_val_true.values)   ** 2)))
test_rmse = float(np.sqrt(np.mean((test_unit.values - y_test_true.values)  ** 2)))

print(f'\n[Refit 완료] (reg 단독, y>0 conditional, transform=none)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5237
[refit fold 4/5] tr_units=20950, vl_units=5237
[refit fold 5/5] tr_units=20950, vl_units=5237

[Refit 완료] (reg 단독, y>0 conditional, transform=none)
  OOF  unit RMSE = 0.006953
  val  unit RMSE = 0.007067
  test unit RMSE = 0.009360


## 6. 산출물 저장

In [6]:
# 산출물 저장: die/unit CSV 6개 + fold_models.pkl + best_params.json.
# postprocess_config=None → 여기선 mean 집계만 (실제 후처리는 combine 단계에서 clf 확률과 곱한 뒤 적용)
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=None,
    study_meta=study_meta,
)

# 저장된 파일 목록
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

# Colab이면 산출물을 zip으로 묶어 로컬 PC로 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'reg_{REG_MODEL_NAME}_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크')
        display(FileLink(_zip))
except ImportError:
    pass

[save_artifacts] c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\03_two_stage\default\reg\catboost\raw\001 저장 완료 (fold_models.pkl + best_params.json + 6 CSV, unit=mean)
  best_params.json                      14.7 KB
  fold_models.pkl                  487,155.2 KB
  oof_die.csv                        5,524.7 KB
  oof_unit.csv                         955.5 KB
  optuna_jh_ts-reg-catboost-raw-001.db       188.0 KB
  test_die.csv                       1,841.4 KB
  test_unit.csv                        318.5 KB
  val_die.csv                        1,841.6 KB
  val_unit.csv                         318.6 KB
